In [1]:
import jax
import os
%env JAX_PLATFORM_NAME=gpu
%env JAX_ENABLE_X64=True
%matplotlib inline
#%matplotlib tk
import numpy as np
import numpy.linalg as lg
#from sklearn.model_selection import train_test_split
import jax
import jax.numpy as jnp
from util_pred import cov_mat, fit_poly_dc, bez_sph, coord_2D3D
from morphomatics.manifold import Sphere, PowerManifold
from morphomatics.stats import ExponentialBarycenter
from TimeSeries.verification_metrics import errfun
from TimeSeries.stats import sph_correlated_trjs, sph_rand_trjs, save_sph, load_sph
from TimeSeries.main import pred
from TimeSeries.model import VelocityEnsemble, Reg, RidgeReg
import matplotlib.pyplot as plt
from random import random
# Set random seed for reproducibility (optional)
np.random.seed(42)

# Global constants and parameters
M = Sphere()
dist = jax.jit(M.metric.dist)
err = errfun(M.metric.dist)

def get_data(type, load, deg=3):
    if load:
        B, Y = load_sph(type)
    else:
        # Reda Data: Random trajectories
        noise_std, mean_curve = 0.05, type  # 'Geo', 'Poly', 'Else'
        n_subj, n_points = 30, 45
        lon_max, lat_max = .75*np.pi, np.pi/20  # 4.5
        Y, template = sph_correlated_trjs(lon_max, lat_max, n_trj=n_subj, n_points=n_points, noise_std=noise_std,mean_curve=mean_curve)  # Gauss noise
        #Y = rand_trjs(n_mat=n_subj, n_points=n_points)
        ##mean_len = int(np.mean([len(Y[i]) for i in range(len(Y))]))
        #B, Y = load_sph(mean_curve)
        B, _ = fit_poly_dc(M, Y, deg=deg)
        #save_sph(B, Y, mean_curve)
        #Y = Y_fit
    return Y, B

ModuleNotFoundError: No module named 'jax'

In [2]:
# Test setting
deg = 3
n_start, n_train = 0, 20
Y, B = get_data(type='Geo', load=False, deg=deg)
Y_train, B_train, Y_test = Y[n_start:n_train], B[n_start:n_train], Y[n_train:]
n_cp = np.shape(B_train[0])[0]
deg, dim = n_cp - 1, 3
P = PowerManifold(M, n_cp)
n_test = len(Y_test)
Ytest = Y_test[5:]
AV_ensemble = VelocityEnsemble(alpha=0.5)
pred_args = {
    'n_learn': n_cp,
    'n_pred': 1,
    'iterative': False
}

In [6]:
# Regression
model = Reg(M, lag=True, degree=deg)
#Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= None)
#Y_pred, MAE_reg, metrics = predx(Ytest, model, n_learn=n_learn, n_pred=1)
#print('MAE: {:.4f}'.format(np.mean(np.concatenate(MAE_reg))))
#mae = [err(Y_pred[k], Ytest[k][n_learn:]).mae() for k in range(len(Ytest))]
#r2_val, acc, mae = validate_pred(Y_test, Y_pred, n_learn=n_learn, n_pred=n_pred)
#model.set_ensemble_strategy(ensemble)
_, metrics_ols = pred(Ytest, model, **pred_args, ensemble_strategy= None)
#Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= AV_ensemble)

MAE: 0.0288


In [3]:
# Ridge regression
# Covariance matrix and mean
Ex = ExponentialBarycenter()
mean_b_train = Ex.compute(P, B_train, max_iter=30)
cov_b_train = cov_mat(P.metric.log, B_train, mean_b_train) + 1e-6*np.eye(n_cp*dim)
#w = jax.vmap(jax.jit(P.metric.log), (None, 0))(mean_b_train, B_train)
#w_vec = w.reshape(n_train, -1)
#cov_b_train, shrinkage = ledoit_wolf(w_vec)
# Analyze correlation
dists_to_mean = [P.metric.dist(mean_b_train, b) for b in B_train]
eigenvalues = np.sort(lg.eigvalsh(cov_b_train))[::-1]
print(f"\nDistance to mean: {np.mean(dists_to_mean):.4f} ± {np.std(dists_to_mean):.4f}")
print(f"Eigenvalues: {eigenvalues}")
print(f"Variance explained by PC2: {eigenvalues[:2].sum()/eigenvalues.sum():.1%}")
#cov_b_train = 1e+6*cov_b_train

# mu=1e-2[1e-4, 1e-2, 1, 5, 10, 20, 40] min at mu = 5*e-2 equal 0.0301
#mu = 4*1e-3  # Else
mu = 1e-2   # Geo
model = RidgeReg(M, mean_b_train, cov_b_train, mu, lag=True, degree=deg)
#Y_pred_new, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= None)
_, metrics_ridge = pred(Ytest, model, **pred_args, ensemble_strategy= None)


Distance to mean: 0.1980 ± 0.0458
Eigenvalues: [1.35173035e-02 7.72595157e-03 4.80596815e-03 4.16685539e-03
 3.29652857e-03 2.52870686e-03 2.03391426e-03 9.69526072e-04
 8.74095117e-04 8.29005970e-04 3.13187234e-04 2.36633075e-04
 1.00000000e-06 1.00000000e-06 1.00000000e-06 1.00000000e-06
 1.00000000e-06 1.00000000e-06]
Variance explained by PC2: 51.4%
MAE: 0.0290
